In [0]:
from pyspark.sql.functions import col, when, current_timestamp

# 1. Четем суровите данни от Bronze слоя
df_bronze_read = spark.table("default.bronze_transactions")

# 2. Правим Silver трансформации (Почистване)
# - Искаме само успешни и чакащи транзакции (без FAILED)
# - Искаме да добавим колона кога са обработени данните
# - Искаме да създадем флаг за "голяма транзакция" (над 200)
df_silver = df_bronze_read \
    .filter(col("status") != "FAILED") \
    .withColumn("ingestion_time", current_timestamp()) \
    .withColumn("is_high_value", when(col("amount") > 200, True).otherwise(False))

# 3. Визуализираме резултата
display(df_silver)

# 4. Записваме като Managed Table в Silver слоя
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("default.silver_transactions")

print("Silver слоят е създаден успешно: default.silver_transactions")